# 2. One Agent with Tools: Investigate a 9.0 Upgrade

### Scenario (continued)
> *Customer upgraded to MongoDB 9.0. Reports: slow aggregations, memory errors on writes, change-stream lag.*

### What this notebook shows
An **agent** is a model that can choose which tools to use. We give it diagnostic tools and a task. The model decides the order.

### The setup on screen
- This notebook on the left
- `9.0 Upcoming.pdf` and `Compatibility changes.pdf` open on the right
- Weather site (open-meteo.com) ready to show

Run cells top to bottom.

## Setup — imports and settings

In [1]:
import ollama, requests, re
# ollama for the LLM, requests for the weather API, re for text splitting.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# TF-IDF for document search (same as notebook 1).

MODEL = "qwen2.5:3b"
MAX_STEPS = 8
# The agent can call tools at most 8 times before we stop it.

OUTPUT_FILE = "support_findings.md"
# The agent will save its diagnostic report to this file.

DOC_PATHS = ["9.0_notes.md", "9.0_compat.md"]
# Two clean markdown sources: release notes + compatibility changes.

print(f"Model: {MODEL}  |  Max steps: {MAX_STEPS}")
print(f"Output: {OUTPUT_FILE}")

Model: qwen2.5:3b  |  Max steps: 8
Output: support_findings.md


## Load all docs (used by the search_docs tool)

In [2]:
doc_texts = {}
# Store each document's text keyed by filename.

for path in DOC_PATHS:
    with open(path, encoding="utf-8") as f:
        doc_texts[path] = f.read()
    print(f"Loaded: {path} ({len(doc_texts[path])} chars)")
# Both .md files loaded — clean text, no PDF artifacts.

Loaded: 9.0_notes.md (7649 chars)
Loaded: 9.0_compat.md (3782 chars)


## Define the diagnostic tools

Each tool is a Python function. The agent can call them — but only the ones we approve.

In [3]:
# ---- TOOL 1: Search the docs for relevant sections ----
def search_docs(query):
    # search_docs uses TF-IDF across all doc files to find relevant text.
    all_chunks = []
    for name, text in doc_texts.items():
        for i in range(0, len(text), 600):
            chunk = text[i:i + 800]
            if len(chunk.strip()) > 50:
                all_chunks.append((name, chunk.strip()))
    if not all_chunks:
        return "No text found in docs."
    vec = TfidfVectorizer(stop_words="english")
    m = vec.fit_transform([c[1] for c in all_chunks] + [query])
    scores = cosine_similarity(m[-1], m[:-1])[0]
    top3 = scores.argsort()[::-1][:3]
    results = []
    for idx in top3:
        doc_name, chunk_text = all_chunks[idx]
        results.append(f"[From: {doc_name}] (score: {scores[idx]:.3f})\n{chunk_text[:500]}")
    return "\n\n---\n\n".join(results)


# ---- TOOL 2: Parse explain() output ----
SAMPLE_EXPLAIN = {
    "explainVersion": "1",
    "queryPlanner": {
        "namespace": "orders.orders",
        "winningPlan": {
            "queryPlan": {"stage": "FETCH", "inputStage": {"stage": "IXSCAN",
                "indexName": "customerId_1_createdAt_1"}},
            "slotBasedPlan": {"slots": "... ixseek ... group ...", "stages": "group [s12 = sum(s8)]"}
        },
        "rejectedPlans": [
            {"queryPlan": {"stage": "COLLSCAN"}},
            {"queryPlan": {"stage": "FETCH", "inputStage": {"stage": "IXSCAN", "indexName": "createdAt_1"}}}
        ]
    },
    "executionStats": {
        "nReturned": 42, "executionTimeMillis": 1847,
        "totalKeysExamined": 1284000, "totalDocsExamined": 1284000,
        "peakTrackedMemBytes": 118734848
    },
    "serverInfo": {"version": "9.0.0"}
}
# This is a realistic explain() output from MongoDB 9.0.

def parse_explain():
    qp = SAMPLE_EXPLAIN["queryPlanner"]
    es = SAMPLE_EXPLAIN["executionStats"]
    wp = qp["winningPlan"]
    engine = "SBE" if "slotBasedPlan" in wp else "Classic"
    # If slotBasedPlan is present, the query used SBE. Otherwise Classic.
    root = wp["queryPlan"]["stage"]
    rejected = [r["queryPlan"]["stage"] for r in qp["rejectedPlans"]]
    selectivity = es["totalDocsExamined"] / max(es["nReturned"], 1)
    # High docsExamined/nReturned ratio means the query scanned a lot.
    mem_mb = es["peakTrackedMemBytes"] / (1024 * 1024)
    return (
        f"Engine: {engine} | Root: {root} | "
        f"Docs examined: {es['totalDocsExamined']} | Returned: {es['nReturned']} | "
        f"Ratio: {selectivity:.0f}x | Time: {es['executionTimeMillis']}ms | "
        f"Peak memory: {mem_mb:.0f}MB | "
        f"Rejected plans: {rejected}"
    )


# ---- TOOL 3: Check $queryStats ----
SAMPLE_QUERYSTATS = {
    "key": {"command": "update", "ns": "orders.orders"},
    "metrics": {
        "cursor": {"firstResponseExecMicros": {"count": 37, "sum": 184000, "max": 19300}},
        "queryExec": {"docsExamined": 984000, "bytesRead": 912345678, "delinquentAcquisitions": 29},
        "queryPlanner": {"fromMultiPlanner": {"true": 31, "false": 6}},
        "writes": {"docsUpdated": 18420, "numMultiUpdates": 37},
        "errors": {"count": 12, "codes": {"146": 8, "292": 4}}
    }
}
# 9.0 $queryStats with the new structure: cursor, queryExec, queryPlanner, writes.

def check_queryStats():
    m = SAMPLE_QUERYSTATS["metrics"]
    scan_ratio = m["queryExec"]["docsExamined"] / max(m["writes"]["docsUpdated"], 1)
    # Ratio of docs examined to docs updated. High ratio = inefficient query.
    errors = m.get("errors", {})
    error_str = ", ".join(f"code {k}: {v} times" for k, v in errors.get("codes", {}).items())
    return (
        f"Operation: {SAMPLE_QUERYSTATS['key']['command']} | "
        f"Examined: {m['queryExec']['docsExamined']} | Updated: {m['writes']['docsUpdated']} | "
        f"Scan ratio: {scan_ratio:.0f}x | Delinquent: {m['queryExec']['delinquentAcquisitions']} | "
        f"Multi-planner runs: {m['queryPlanner']['fromMultiPlanner']['true']}/37 | "
        f"Errors: {error_str if error_str else 'none'}"
    )


# ---- TOOL 4: Read serverStatus ----
SAMPLE_SERVERSTATUS = {
    "host": "rs0-primary-0",
    "version": "9.0.0",
    "metrics": {
        "query": {
            "operationsFailedDueToMemoryLimit": 137,
            "configuredMaxMemoryUsageBytesPerOperation": 1073741824,
            "operationsSpilledToDisk": 4812,
            "lookupUnwind": 8420
        },
        "operation": {
            "writeConflictRetryLimitHit": 53
        }
    },
    "opcounters": {"insert": 1200, "query": 45000, "update": 8200, "delete": 300}
}
# Real 9.0 serverStatus with new memory guardrail and lookupUnwind counters.

def read_serverStatus():
    ss = SAMPLE_SERVERSTATUS
    mq = ss["metrics"]["query"]
    max_mem_gb = mq["configuredMaxMemoryUsageBytesPerOperation"] / (1024**3)
    return (
        f"Host: {ss['host']} | MongoDB {ss['version']} | "
        f"Memory limit: {max_mem_gb:.1f}GB | "
        f"Ops failed (memory): {mq['operationsFailedDueToMemoryLimit']} | "
        f"Spilled to disk: {mq['operationsSpilledToDisk']} | "
        f"$lookup+$unwind in SBE: {mq['lookupUnwind']} calls | "
        f"Write conflicts: {ss['metrics']['operation']['writeConflictRetryLimitHit']} | "
        f"Ops/sec: {sum(ss['opcounters'].values())}"
    )


# ---- TOOL 5: Classify error codes ----
ERROR_CODES = {
    146: ("ExceededMemoryLimit", "Query exceeded the 9.0 per-operation memory guardrail (default 1GB or 20% RAM). Check explain() peakTrackedMemBytes."),
    292: ("QueryExceededMemoryLimitNoDiskUseAllowed", "Memory-intensive operation cannot spill because allowDiskUse is false. Enable disk use or add indexes."),
    509: ("TooManyOpenTransactions", "Too many concurrent multi-doc transactions (default limit: 10,000). Check for unclosed transactions."),
    485: ("InterruptedDueToTimeseriesUpgradeDowngrade", "Direct access to legacy time-series bucket namespace. 9.0 uses single namespace."),
    491: ("CommandNotSupportedOnLegacyTimeseriesBucketsNamespace", "Use the logical collection name, not system.buckets."),
}
# Real MongoDB 9.0 error codes from error_codes.yml.

def classify_error(code):
    info = ERROR_CODES.get(int(code))
    if info:
        return f"Code {code} ({info[0]}): {info[1]}"
    return f"Code {code}: Unknown — check MongoDB error code docs."


# ---- TOOL 6: Get real weather (Open-Meteo, no API key) ----
def get_weather(city="Dublin"):
    coords = {"Dublin": (53.3498, -6.2603), "London": (51.5074, -0.1278),
               "New York": (40.7128, -74.0060), "Sydney": (-33.8688, 151.2093)}
    lat, lon = coords.get(city, coords["Dublin"])
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {"latitude": lat, "longitude": lon, "current": "temperature_2m,precipitation,wind_speed_10m"}
        d = requests.get(url, params=params, timeout=10).json()["current"]
        return (f"{city} now: {d['temperature_2m']}C, "
                f"precipitation {d['precipitation']}mm, wind {d['wind_speed_10m']}km/h")
    except Exception as e:
        return f"Weather request failed: {e}"


# ---- TOOL 7: Save the diagnostic report ----
def save_report(content):
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Report saved to {OUTPUT_FILE}"


# Quick tool check
print("--- Tool checks ---")
print("search_docs:", search_docs("per-operation memory limit")[:100], "...")
print("parse_explain:", parse_explain())
print("check_queryStats:", check_queryStats())
print("read_serverStatus:", read_serverStatus())
print("classify_error 146:", classify_error(146))
print("get_weather:", get_weather("Dublin"))

--- Tool checks ---
search_docs: [From: 9.0_notes.md] (score: 0.450)
of memory that a single query operation can use. By default, the ...
parse_explain: Engine: SBE | Root: FETCH | Docs examined: 1284000 | Returned: 42 | Ratio: 30571x | Time: 1847ms | Peak memory: 113MB | Rejected plans: ['COLLSCAN', 'FETCH']
check_queryStats: Operation: update | Examined: 984000 | Updated: 18420 | Scan ratio: 53x | Delinquent: 29 | Multi-planner runs: 31/37 | Errors: code 146: 8 times, code 292: 4 times
read_serverStatus: Host: rs0-primary-0 | MongoDB 9.0.0 | Memory limit: 1.0GB | Ops failed (memory): 137 | Spilled to disk: 4812 | $lookup+$unwind in SBE: 8420 calls | Write conflicts: 53 | Ops/sec: 54700
classify_error 146: Code 146 (ExceededMemoryLimit): Query exceeded the 9.0 per-operation memory guardrail (default 1GB or 20% RAM). Check explain() peakTrackedMemBytes.
get_weather: Dublin now: 14.1C, precipitation 0.0mm, wind 13.7km/h


## Describe the tools to the model

The model needs to know what tools exist. `TOOLS` is our safety list — only these names can run.

In [4]:
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "search_docs",
        "description": "Search the MongoDB 9.0 docs for relevant text about a topic.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "parse_explain",
        "description": "Parse the customer's explain() output. Returns engine (SBE/Classic), plan root, docs examined, memory, rejected plans.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "check_queryStats",
        "description": "Check $queryStats for the customer's workload. Shows write patterns, scan ratios, errors.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "read_serverStatus",
        "description": "Read serverStatus metrics. Shows memory guardrails, spill counts, SBE usage, write conflicts.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "classify_error",
        "description": "Classify a MongoDB error code. Returns name and explanation.",
        "parameters": {"type": "object",
            "properties": {"code": {"type": "integer"}},"required": ["code"]}}},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city. Shows real-time API call.",
        "parameters": {"type": "object",
            "properties": {"city": {"type": "string"}}}}},
    {"type": "function", "function": {
        "name": "save_report",
        "description": "Save the final diagnostic report to support_findings.md.",
        "parameters": {"type": "object",
            "properties": {"content": {"type": "string"}},"required": ["content"]}}},
]
# These schemas tell the model what each tool does and what arguments it needs.

TOOLS = {
    "search_docs": search_docs,
    "parse_explain": parse_explain,
    "check_queryStats": check_queryStats,
    "read_serverStatus": read_serverStatus,
    "classify_error": classify_error,
    "get_weather": get_weather,
    "save_report": save_report,
}
# Safety guard: only these functions can run, no matter what the model says.

print("Allowed tools:", list(TOOLS))

Allowed tools: ['search_docs', 'parse_explain', 'check_queryStats', 'read_serverStatus', 'classify_error', 'get_weather', 'save_report']


## The task — what we ask the agent to do

In [6]:
TASK = (
    "You are investigating a MongoDB 9.0 upgrade case. Customer reports: "
    "(1) slow aggregations after upgrade, "
    "(2) memory errors on writes (error codes 146 and 292), "
    "(3) change-stream lag. "
    "\n\nYour job:\n"
    "1. Search the 9.0 docs for what changed that could cause these symptoms.\n"
    "2. Use parse_explain() to check if SBE is being used.\n"
    "3. Use check_queryStats() to see write patterns and errors.\n"
    "4. Use read_serverStatus() to check memory guardrails.\n"
    "5. Classify any error codes you find.\n"
    "6. Check the weather (for fun — shows real API call).\n"
    "7. Save a structured diagnostic report with: "
    "Symptom, Likely 9.0 Change, Evidence, Risk Level, Recommendation."
)
print(TASK)

You are investigating a MongoDB 9.0 upgrade case. Customer reports: (1) slow aggregations after upgrade, (2) memory errors on writes (error codes 146 and 292), (3) change-stream lag. 

Your job:
1. Search the 9.0 docs for what changed that could cause these symptoms.
2. Use parse_explain() to check if SBE is being used.
3. Use check_queryStats() to see write patterns and errors.
4. Use read_serverStatus() to check memory guardrails.
5. Classify any error codes you find.
6. Check the weather (for fun — shows real API call).
7. Save a structured diagnostic report with: Symptom, Likely 9.0 Change, Evidence, Risk Level, Recommendation.


## The agent loop

This is the heart of an agent:
1. Send conversation + tool list to the model
2. If model asks for a tool → run it → feed result back
3. If model gives final answer → stop
4. Never loop more than MAX_STEPS times

In [7]:
messages = [
    {"role": "system", "content":
        "You are a MongoDB support engineer agent. Use the tools to investigate. "
        "IMPORTANT: When done, you MUST call save_report with your full findings. "
        "Do not just talk about calling it — actually call save_report(). "
        "Then give a short 1-line confirmation."},
    {"role": "user", "content": TASK},
]
# Start the conversation.

final_text = ""
# Collect the model's final text in case save_report isn't called.

for step in range(MAX_STEPS):
    reply = ollama.chat(model=MODEL, messages=messages, tools=TOOL_SCHEMAS)
    msg = reply["message"]
    messages.append(msg)

    tool_calls = msg.get("tool_calls")
    text = msg.get("content", "")
    if text.strip():
        final_text = text.strip()
        # Keep track of the last meaningful text.

    if not tool_calls:
        if text.strip():
            print("\n=== AGENT FINAL REPLY ===\n")
            print(text.strip())
            break
        print("\n  (nudging model to finish...)")
        messages.append({"role": "user", "content":
            "Call save_report with your findings NOW. Do not explain, just call it."})
        continue

    for call in tool_calls:
        name = call["function"]["name"]
        args = call["function"].get("arguments", {})
        # args might be a JSON string from some Ollama versions
        if isinstance(args, str):
            import json
            args = json.loads(args) if args.strip() else {}
        if not isinstance(args, dict):
            args = {}
        print(f"\nStep {step+1}: -> {name}")
        if args:
            print(f"  args: {args}")
        result = TOOLS[name](**args) if args else TOOLS[name]()
        print(f"  result: {str(result)[:150]}")
        messages.append({"role": "tool", "content": str(result)})
else:
    print(f"\nReached max steps. Forcing final summary...")
    messages.append({"role": "user", "content": "Summarize findings in 3 bullet points."})
    final = ollama.chat(model=MODEL, messages=messages)
    final_text = final["message"]["content"].strip()
    print("\n=== FORCED SUMMARY ===\n")
    print(final_text)

# Fallback: if save_report wasn't called, save the final text
import os
if not os.path.exists(OUTPUT_FILE) and final_text:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(final_text)
    print(f"\n(fallback: saved model text to {OUTPUT_FILE})")


Step 1: -> search_docs
  args: {'query': 'upgrade 9.0 slow aggregations memory errors 292 146 change-stream lag'}
  result: [From: 9.0_notes.md] (score: 0.164)
of memory that a single query operation can use. By default, the limit is 1 gigabyte or 20% of the memory availabl

Step 1: -> parse_explain
  result: Engine: SBE | Root: FETCH | Docs examined: 1284000 | Returned: 42 | Ratio: 30571x | Time: 1847ms | Peak memory: 113MB | Rejected plans: ['COLLSCAN', '

Step 1: -> check_queryStats
  result: Operation: update | Examined: 984000 | Updated: 18420 | Scan ratio: 53x | Delinquent: 29 | Multi-planner runs: 31/37 | Errors: code 146: 8 times, code

Step 1: -> read_serverStatus
  result: Host: rs0-primary-0 | MongoDB 9.0.0 | Memory limit: 1.0GB | Ops failed (memory): 137 | Spilled to disk: 4812 | $lookup+$unwind in SBE: 8420 calls | Wr

Step 1: -> classify_error
  args: {'code': 146}
  result: Code 146 (ExceededMemoryLimit): Query exceeded the 9.0 per-operation memory guardrail (default 1G

## See what the agent saved

In [8]:
try:
    with open(OUTPUT_FILE, encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    print(f"{OUTPUT_FILE} not found.")

Symptom: Slow aggregations, Likely 9.0 Change: SBE use, Evidence: parse_explain(), Risk Level: High, Recommendation: Review SBE usage and disable if necessary.
Symptom: Memory errors, Likely 9.0 Change: 9.0 write patterns, Evidence: check_queryStats(), Risk Level: Medium, Recommendation: Investigate and adjust write patterns.
Symptom: Change-stream lag, Likely 9.0 Change: Change-stream design, Evidence: read_serverStatus(), Risk Level: Low, Recommendation: Monitor and optimize change-stream performance.
Symptom: Slow writes, Likely 9.0 Change: 9.0 write patterns, Evidence: check_queryStats(), Risk Level: High, Recommendation: Adjust write patterns and monitor.
Symptom: Weather, Evidence: get_weather(), Risk Level: Not applicable, Recommendation: Enjoy the weather.


## Recap

- The model **chose** which tools to use and in what order
- We ran only the 7 approved tools — safety by design
- The agent diagnosed a realistic 9.0 upgrade case
- That decision-making loop is what makes it an **agent**, not just RAG

**Next:** Notebook 3 splits this into two specialized agents — Investigator + TS Reviewer.